In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import random
import sys
sys.path.append("/home/dev/pose/pose")

import yaml

WORK_DIR = "/home/dev/pose"

In [20]:
%matplotlib inline
import math
import torch

from matplotlib import pyplot as plt
from torchvision.io import read_image
from torchvision.transforms.v2 import InterpolationMode
from torchvision.transforms.v2 import ToPILImage
from torchvision.transforms.v2.functional import affine

from torchvision.utils import draw_bounding_boxes
from torchvision.utils import draw_keypoints

to_pil = ToPILImage()

COLOR_MAGENTA = (255, 0, 255) # RGB
COLOR_CYAN = (0, 255, 255) # RGB
COLOR_YELLOW = (255, 255, 0) # RGB

In [3]:
config_fname = "/home/dev/pose/configs/resnet50_simcc-tr.yaml"

with open(config_fname) as config_fin:
    cfg = yaml.load(config_fin, Loader=yaml.Loader)

In [4]:
from data.dataset import CocoDataset

In [25]:
connectivity = [[15, 13], [13, 11], [16, 14], [14, 12], [11, 12], [5, 11],
                [6, 12], [5, 6], [5, 7], [6, 8], [7, 9], [8, 10], [1, 2],
                [0, 1], [0, 2], [1, 3], [2, 4], [3, 5], [4, 6]]

In [5]:
dataset = CocoDataset(os.path.join(WORK_DIR, cfg['train_data']['anno_fname']),
                      os.path.join(WORK_DIR, cfg['train_data']['image_dir']))

2024-12-18 13:37:55.896 | INFO     | data.dataset:init_coco_info:78 - Keypoints idx2name: dict_items([(0, 'nose'), (1, 'L-eye'), (2, 'R-eye'), (3, 'L-ear'), (4, 'R-ear'), (5, 'L-shder'), (6, 'R-shder'), (7, 'L-elb'), (8, 'R-elb'), (9, 'L-wrst'), (10, 'R-wrst'), (11, 'L-hip'), (12, 'R-hip'), (13, 'L-knee'), (14, 'R-knee'), (15, 'L-ankl'), (16, 'R-ankle')])
2024-12-18 13:37:57.680 | INFO     | data.dataset:__init__:26 - Annotation fname: /home/dev/pose/data/coco/annotations/person_keypoints_train2017.json
2024-12-18 13:37:57.751 | INFO     | data.dataset:__init__:38 - Num images: 118,287
2024-12-18 13:37:57.751 | INFO     | data.dataset:__init__:41 - Num raw annotations: 262,465
2024-12-18 13:38:01.031 | INFO     | data.dataset:__init__:64 - skip crowd: 5,212
2024-12-18 13:38:01.031 | INFO     | data.dataset:__init__:65 - skip not human: 0
2024-12-18 13:38:01.031 | INFO     | data.dataset:__init__:66 - skip insufficient keypoints: 107,440
2024-12-18 13:38:01.032 | INFO     | data.dataset

In [131]:
sample = dataset.get_sample(0)
image_meta = dataset.get_image_meta(sample['image_id'])
image_path = image_meta['file_name']

In [12]:
sample.keys()

dict_keys(['keypoints', 'bbox', 'image_id', 'id'])

In [132]:
image_meta.keys()

dict_keys(['file_name', 'height', 'width'])

In [63]:
def label_affine(kp, angle, bbox_center):
    theta = -1 * math.pi * angle / 180
    M = [[math.cos(theta), -1 * math.sin(theta)],
         [math.sin(theta), math.cos(theta)]]
    M = torch.tensor(M)

    return torch.matmul((kp[:,:2] - bbox_center).float(), M).int() + bbox_center

In [157]:
def adjust_bbox(kp, height, width):
    _kp = kp[kp[:,2] > 0]
    x_max = _kp[:,0].max()
    x_min = _kp[:,0].min()
    y_max = _kp[:,1].max()
    y_min = _kp[:,1].min()
    pad = 15
    bbox = [x_min-pad, y_min-pad, x_max+pad, y_max+pad]
    print(f"original tr bbox: {bbox}")
    bbox = torch.tensor([[max(x_min-pad, 0),
                          max(y_min-pad, 0),
                          min(x_max+pad, width),
                          min(y_max+pad, height)]])
    return bbox

In [160]:
def test():
    sample_idx = random.randint(0, len(dataset)-1)
    print(f"Sample index: {sample_idx}")
    sample = dataset.get_sample(sample_idx)
    image_meta = dataset.get_image_meta(sample['image_id'])
    image_path = image_meta['file_name']
    print(f"Image path: {image_path}")
    height = image_meta['height']
    width = image_meta['width']
    print(f"H:{height}, W:{width}")

    bbox = sample['bbox']
    bbox_4p = torch.tensor([[bbox[0], bbox[1], bbox[0]+bbox[2], bbox[1]+bbox[3]]])
    print(f"bbox: {bbox}")

    kp = sample['keypoints']
    kp_formatted = kp[:,:2].reshape(1,17,2)
    visibility = kp[:,2].reshape(1,17)
    print(f"Keypoints: {kp}")

    img = read_image(image_path)
    img = draw_bounding_boxes(img, bbox_4p, colors=COLOR_MAGENTA, width=5)
    img = draw_keypoints(img, kp_formatted, connectivity=connectivity, colors=COLOR_MAGENTA, radius=10, visibility=visibility)

    angle = random.randint(-45,45)    # angle in degree: -180 to 180
    print(f"angle: {angle}")
    translate = [0, 0]  # horizontal and vertical translations
    scale = 1.0         # overall scale
    shear = [0, 0]
    interpolation = InterpolationMode.BILINEAR
    fill = None
    center = [int(bbox[0] + bbox[2]/2), int(bbox[1] + bbox[3]/2)]
    
    img1 = affine(img, angle, translate, scale, shear, interpolation, fill, center)
    kp_transformed = label_affine(kp, angle, torch.tensor(center))
    print(f"Transformed keypoints: {kp_transformed}")
    ad_bbox = adjust_bbox(kp_transformed, height, width)
    print(f"adjust bbox(4points format): {ad_bbox}")
    img1 = draw_bounding_boxes(img1, ad_bbox, colors=COLOR_CYAN, width=3)
    img1 = draw_keypoints(img1, kp_transformed.reshape(1,17,2), connectivity=connectivity, colors=COLOR_CYAN, radius=5, visibility=visibility)
    p_img = to_pil(img1)
    plt.imshow(p_img)
    plt.show()

In [161]:
test()

Sample index: 123193
Image path: /home/dev/pose/data/coco/images/train2017/000000034632.jpg
H:427, W:640
bbox: [45, 144, 26, 85]
Keypoints: tensor([[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [ 52, 154,   2],
        [  0,   0,   0],
        [ 51, 161,   2],
        [ 60, 162,   2],
        [ 50, 175,   2],
        [  0,   0,   0],
        [ 49, 184,   2],
        [  0,   0,   0],
        [ 53, 184,   2],
        [ 61, 184,   2],
        [ 53, 202,   2],
        [ 59, 201,   2],
        [ 57, 222,   2],
        [ 62, 222,   2]], dtype=torch.int32)
angle: 39
Transformed keypoints: tensor([[129,   5],
        [129,   5],
        [129,   5],
        [ 73, 158],
        [129,   5],
        [ 68, 163],
        [ 74, 169],
        [ 58, 173],
        [129,   5],
        [ 53, 179],
        [129,   5],
        [ 56, 182],
        [ 61, 186],
        [ 45, 195],
        [ 50, 198],
        [ 35, 213],
        [ 39, 216]])


IndexError: index 2 is out of bounds for dimension 1 with size 2

In [151]:
x= torch.tensor([[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [460, 241,   2],
        [401, 232,   2],
        [446, 316,   2],
        [396, 280,   2],
        [423, 364,   2],
        [362, 278,   2],
        [  0,   0,   0],
        [378, 357,   2],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]])

In [152]:
x

tensor([[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [460, 241,   2],
        [401, 232,   2],
        [446, 316,   2],
        [396, 280,   2],
        [423, 364,   2],
        [362, 278,   2],
        [  0,   0,   0],
        [378, 357,   2],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0]])

In [156]:
x[x[:,2] > 0]

tensor([[460, 241,   2],
        [401, 232,   2],
        [446, 316,   2],
        [396, 280,   2],
        [423, 364,   2],
        [362, 278,   2],
        [378, 357,   2]])